https://medium.com/@sauravjoshi23/complex-query-resolution-through-llamaindex-utilizing-recursive-retrieval-document-agents-and-sub-d4861ecd54e6

# Dataset Loader

In [ ]:
import os, re
from typing import Optional, Dict, List
from pathlib import Path
from llama_index.readers.base import BaseReader
from llama_index import Document

class XMLTagReader(BaseReader):
    """
    Read XML files and extract text from a specific tag with BeautifulSoup.
    By default, reads the text from the ``<TEXT>`` tag.
    """
    def __init__(
        self,
        tag: str = "TEXT"
    ) -> None:
        self._tag = tag
        super().__init__()

    def load_data(
        self, file: Path, extra_info: Optional[Dict] = None
    ) -> List[Document]:
        try:
            from bs4 import BeautifulSoup
        except ImportError:
            raise ImportError("bs4 is required to read HTML files.")
        files = []
        if os.path.isdir(file):
            for path, subdirs, fs in os.walk(file):
                for name in fs:       
                    files.append(os.path.join(path, name))
        elif os.path.isfile(file):
            files = [file]
        else:
            raise Exception("Not support file type")
        docs = []
        for file in files:   
            with open(file, encoding="utf-8") as xml_file:
                soup = BeautifulSoup(xml_file, "xml")

            tags = soup.find_all(self._tag)
            for t in tags:
                tag_text = self._extract_text_from_tag(t)

                x = re.search(r"(\d{4}-\d{2}-\d{2}(_\d)?)", str(file))
                date = x.group(1)
                metadata = {
                    "tag": self._tag,
                    "file_path": str(file),
                    "date": date
                }
                metadata.update(extra_info or {})

                doc = Document(
                    text=tag_text,
                    metadata=metadata,
                )
                docs.append(doc)
        return docs

    def _extract_text_from_tag(self, tag) -> str:
        try:
            from bs4 import NavigableString
        except ImportError:
            raise ImportError("bs4 is required to read HTML files.")

        texts = []
        for elem in tag.children:
            if isinstance(elem, NavigableString):
                if elem.strip():
                    texts.append(elem.strip())
            elif elem.name == self._tag:
                continue
            else:
                texts.append(elem.get_text().strip())
        return "\n".join(texts)
    
# reader = XMLTagReader()
# docs = reader.load_data(
#     "lung_cancer/58/path/58_2020-07-14.xml"
# )
# print(docs[0].text)

In [ ]:
import os, re
from typing import Optional, Dict, List
from pathlib import Path
from llama_index.readers.base import BaseReader
from llama_index import Document

class PlainTextReader(BaseReader):
    """
    Read plain text files and extract text.
    """
    def __init__(
        self
    ) -> None:
        super().__init__()

    def load_data(
        self, file: Path, extra_info: Optional[Dict] = None
    ) -> List[Document]:
        
        files = []
        if os.path.isdir(file):
            for path, subdirs, fs in os.walk(file):
                for name in fs:       
                    files.append(os.path.join(path, name))
        elif os.path.isfile(file):
            files = [file]
        else:
            raise Exception("Not support file type")
        docs = []
        for file in files:   
            with open(file, encoding="utf-8") as txt_file:
                text = "\n".join(txt_file.readlines())
                x = re.search(r"(\d{4}-\d{2}-\d{2}(_\d)?)", str(file))
                date = x.group(1)
                metadata = {
                    "file_path": str(file),
                    "date": date
                }
                metadata.update(extra_info or {})

                doc = Document(
                    text=text,
                    metadata=metadata,
                )
                docs.append(doc)
        return docs    
reader = PlainTextReader()
# docs = reader.load_data(
#     "lung_cancer/202209000226/2021-08-24.txt"
#     #"lung_cancer/202212000218/2021-10-15.txt"
    
# )
# print(docs[0].text)

sample_path = os.path.join("lung_cancer", "202209000226")
#sample_path = os.path.join("lung_cancer", "202212000218")
docs = reader.load_data(sample_path)
docs

In [ ]:
import os, re
from typing import Optional, Dict, List
from pathlib import Path
from llama_index.readers.base import BaseReader
from llama_index import Document

class CRCPlainTextReader(BaseReader):
    """
    Read plain text files and extract text.
    """
    def __init__(
        self
    ) -> None:
        super().__init__()

    def load_data(
        self, file: Path, extra_info: Optional[Dict] = None
    ) -> List[Document]:
        
        files = []
        if os.path.isdir(file):
            for path, subdirs, fs in os.walk(file):
                for name in fs:       
                    files.append(os.path.join(path, name))
        elif os.path.isfile(file):
            files = [file]
        else:
            raise Exception("Not support file type")
        docs = []
        for file in files:   
            with open(file, encoding="utf-8") as txt_file:
                text = "\n".join(txt_file.readlines())
                x = re.search(r"(\d{4}-\d{2}-\d{2}(_\d)?)", str(file))
                date = x.group(1)
                metadata = {
                    "file_path": str(file),
                    "date": date
                }
                metadata.update(extra_info or {})

                doc = Document(
                    text=text,
                    metadata=metadata,
                )
                docs.append(doc)
        return docs    
reader = CRCPlainTextReader()

sample_path = os.path.join("lung_cancer", "202209000226")
#sample_path = os.path.join("lung_cancer", "202212000218")
docs = reader.load_data(sample_path)
docs

# Document Agent and Vector Index Engine based on Auto-merging Retrieval

In [ ]:
from llama_index.prompts.base import PromptTemplate
from llama_index.llms import HuggingFaceLLM
import torch
from transformers import BitsAndBytesConfig

#selected_model = "meta-llama/Llama-2-7b-chat-hf"
selected_model = "mistralai/Mistral-7B-Instruct-v0.1"

system_prompt = '''As a lung cancer registrar in this system, your primary responsibility is to accurately curate lung cancer registry data in compliance with user-defined requirements. It's imperative that your data compilation strictly adheres to the options and categories specified by users. In situations where the correct code cannot be determined due to insufficient information, or the content is unrelated to lung cancer, your response should be 'NA'. All responses must be formatted exclusively as valid JSON objects. Outputs in any format other than JSON are not acceptable. Each response should include the key 'explan', containing a string that justifies your classification choice. Additionally, incorporate other keys as required by user specifications. Remember, the structure and content of your JSON response should reflect the precision and adherence to guidelines essential for this role.'''

query_wrapper_prompt = PromptTemplate(
    "[INST]<<SYS>>\n" + system_prompt + "<</SYS>>\n\n{query_str}[/INST]"
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

doc_idx_llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=2048,    
    generate_kwargs={"temperature": 0.001, "do_sample": True},
    #query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name=selected_model,
    model_name=selected_model,
    device_map="auto", 
    system_prompt = system_prompt,   
    query_wrapper_prompt=query_wrapper_prompt,
    model_kwargs={"quantization_config": quantization_config, "pad_token_id" : 2},
)

In [ ]:
from llama_index.node_parser import HierarchicalNodeParser

node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128]
)

nodes = node_parser.get_nodes_from_documents([docs[0]])
print(len(nodes)) # 以這個例子來說，最上層是一個 node，下面有兩個 node，在往下第一個子節點有 5 個子節點，第二個子節點有三個子節點

from llama_index.node_parser import get_leaf_nodes

# 總共 8 個子節點
leaf_nodes = get_leaf_nodes(nodes)
print(len(leaf_nodes))

In [ ]:
for leaf in leaf_nodes:
    print(len(leaf.text.split('\n')))
    for text in leaf.text.split('\n'):
        print(text)
    print(leaf.parent_node.node_id)
    print("-"*50)
print("="*50)    
leafs_by_id = {node.node_id: node for node in leaf_nodes}
for ids in leafs_by_id:    
    parent_node = leafs_by_id[ids].parent_node
    print(f'{ids} --> {parent_node.node_id}')
print(parent_node.parent_node.node_id) # 到頂了

# Construct Index

## Agent Service Context

In [ ]:
from llama_index.llms.llama_utils import messages_to_prompt, completion_to_prompt
from llama_index.llms import HuggingFaceLLM
from llama_index import ServiceContext

agent_system_prompt = '''As a lung cancer registrar agent, your primary responsibility is to accurately curate lung cancer registry data in compliance with user-defined requirements and the document you manage. It's imperative that your data compilation strictly adheres to the options and categories specified by users. In situations where the correct code cannot be determined due to insufficient information, or the content is unrelated to lung cancer, your response should be 'NA'. All responses must be formatted exclusively as valid JSON objects. Outputs in any format other than JSON are not acceptable. Each response should include the key 'explan', containing a string that justifies your classification choice. Additionally, incorporate other keys as required by user specifications. Remember, the structure and content of your JSON response should reflect the precision and adherence to guidelines essential for this role.'''

agent_llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=2048,    
    generate_kwargs={"temperature": 0.001, "do_sample": True},
    tokenizer_name=selected_model,
    model_name=selected_model,
    is_chat_model = True,

    system_prompt=agent_system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    
    
    # Here is how to fix the alternative error show above.
    # We transform inputs into Llama2 format here
    messages_to_prompt = messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    device_map="auto", 
    model_kwargs={"quantization_config": quantization_config, "pad_token_id" : 2},
)

agent_service_context = ServiceContext.from_defaults(
    llm=agent_llm,
    embed_model="local:BAAI/bge-large-en-v1.5"
)

## Query Engine

See https://platform.openai.com/docs/guides/prompt-engineering?utm_source=ai.briefnewsletter.com&utm_medium=newsletter&utm_campaign=2023-ai

In [ ]:
# llama_template_laterality ='''Your task is to determine the laterality of lung cancer based on the primary tumor site. The laterality refers to whether the cancer originates from a paired organ and is applicable only to primary tumors. When responding, please use the following numerical options:

# - 0: The cancer does not originate from a paired organ.
# - 1: The primary origin of the cancer is on the right side.
# - 2: The primary origin of the cancer is on the left side.

# Note: Ensure that your response is limited to one of these three options. For instance, if the primary tumor site is in the right lung, your response must be "1". After providing the code, please add two newline characters and then explain the reason for your classification choice.'''

# llama_template_primary_site ='''Your task is to identify and confirm the primary site of lung cancer. The primary site usually refers to specific regions within the respiratory system. When providing your response, please choose from the following options, which correspond to the standard coding for different lung cancer sites:

# - C339: Trachea
# - C340: Main bronchus
# - C341: Upper lobe, lung
# - C342: Middle lobe, lung
# - C343: Lower lobe, lung
# - C348: Overlapping lesion of lung
# - C349: Lung NOS (Not Otherwise Specified)

# Your response should be limited to one of these codes, based on the most accurate location of the primary cancer site. For example, if the primary site is in the upper lobe of the lung, your response must be 'C341'. After providing the code, please add two newline characters and then explain the reason for your classification choice.'''

llama_template_histology ='''Your primary task is to accurately curate the histology code for patients diagnosed with lung cancer. This involves meticulously reviewing and analyzing the given medical record of the patient to determine the most appropriate histology code. Your decision should be based on a thorough assessment of clinical findings and any relevant medical descriptions that shed light on the histological nature of the lung cancer. It is essential to use only the information provided in the document at hand.
When responding, you are required to select the most appropriate option from the following standard codes, which represents different histology types of lung cancer:

- 8550: Use this code if the histology type of the lung cancer case is "acinar cell carcinoma", "acinar adenocarcinoma", "acinar carcinoma", or "acinic cell adenocarcinoma".
- 8551: Use this code if the histology type is "acinar cell cystadenocarcinoma" or "acinar adenocarcinoma of the lung".
- 8041: Use this code if the histology type is "small cell carcinoma, pulmonary type", "small cell neuroendocrine carcinoma", "round cell carcinoma", "reserve cell carcinoma", or "small cell carcinoma, NOS"
- 8070: Use this code if the histology type is "squamous cell carcinoma in situ, NOS", "epidermoid carcinoma in situ, NOS", "intraepidermal carcinoma, NOS", "intraepithelial squamous cell carcinoma", "squamous cell carcinoma, NOS", "squamous cell carcinoma, usual type", "epidermoid carcinoma, NOS", "squamous carcinoma", or "squamous cell epithelioma"

Your response should be limited to one of these codes, aligning with the most accurate histological classification of the lung cancer case. For instance, if the case is diagnosed as 'Acinar adenocarcinoma of the lung', your response must be
{
"explan": "[Insert your explanation here based on the document]",
"cite": "[Insert the relevant passages extracted from the document used for your decision]",
"histology": "8551"
}'''

llama_template_primary_site_laterality ='''Your task as an assistant is to identify and confirm the primary site and laterality of lung cancer. The primary site refers to specific regions within the respiratory system. The laterality refers to whether the cancer originates from a paired organ and is applicable only to primary tumors. It is essential to use only the information provided in the document at hand. Choose from the following standard codes for lung cancer sites and laterality:

Primary site codes:
- C339: Trachea
- C340: Main bronchus
- C341: Upper lobe, lung
- C342: Middle lobe, lung
- C343: Lower lobe, lung
- C348: Overlapping lesion of lung
- C349: Lung NOS (Not Otherwise Specified)

Laterality codes:
- 0: Cancer does not originate from a paired organ.
- 1: Primary origin of the cancer is on the right side.
- 2: Primary origin of the cancer is on the left side.

Your response must be a valid JSON object containing the following keys:
-'primary site': A string containing the code for the primary site.
-'laterality': A string containing the code for laterality.

Ensure your response is limited to the provided options for primary site and laterality. For instance, if the primary site and laterality are in the 'upper lobe' of the 'right' lung, your JSON response should be:
{
"explan": "[Insert your explanation here based on the document]",
"cite": "[Insert the relevant passages extracted from the document used for your decision]",
"primary site": "C341",
"laterality": "1"
}'''

# llama_template_behavior = '''Your task is to record the behavior code in the pathology diagnosis. The behavior code is represented by the 5th code in the M-code system. When providing your response, please choose from the following options, which correspond to the standard behavior coding used by pathologist:

# - N/A: Use this code if there is no information provided in the pathology report.
# - 0: Benign
# - 1: Borderline
# - 2: In situ
# - 3: Malignant, primary site
# - 6: Malignant, metastatic site
# - 9: Malignant, uncertain whether primary or metastatic site

# Please ensure your response is limited to one of these specific codes. For example, if a tumor's code is M-8070/3, your response must be '3' indicaing that the tumor is malignant and located at the primary site, as denoted by the /3 behavior code. After providing the code, please add two newline characters and then explain the reason for your classification choice.'''

# llama_template_node_examine = '''Your task is to record the total number of regional lymph nodes examined by the pathologist in each case. This data is crucial for understanding the extent of lymph node examination in cancer cases. When coding, focus on the total count of lymph nodes examined, regardless of whether they are positive or negative for malignancy.

# Here are two examples to guide you:

# Example 1: The report states 'Lymph node, regional, lymphadenectomy, negative for malignancy (0/20)'. The code for this case is '20', representing the total number of lymph nodes examined.
# Example 2: The report says 'Lymph node, apical, lymphadenectomy, negative for malignancy (0/9)'. In this case, the correct code is '9', indicating the total count of lymph nodes examined.
# After providing the code, add two newline characters and then provide a detailed explanation for your coding decision. This explanation should include the rationale behind selecting the specific count based on the information provided in the pathology report.'''

# llama_template_node_positive='''Your task is to record the total number of regional lymph nodes that tested positive for malignancy as examined by the pathologist. This information is crucial for assessing the extent of cancer involvement in lymph nodes.
# When providing your answer, focus on the count of lymph nodes that were positive for malignancy. Here are two examples to illustrate the process:
# Example 1: The pathology report states 'Lymph node, regional, lymphadenectomy, negative for malignancy (12/20)'. The answer in this case should be '12', as there were 12 lymph nodes testing positive for malignancy out of the 20 examined.
# Example 2: The report reads 'Lymph node, apical, lymphadenectomy, negative for malignancy (0/9)'. Here, the correct answer is also '0', indicating that none of the 9 examined lymph nodes were positive for malignancy.
# After providing the answer, add two newline characters and then provide a detailed explanation of your reasoning. This explanation should clearly articulate why you selected the specific count based on the pathology report's findings.'''

# llama_template_path_TNM = '''Your task is to response the cancer staging information in terms of PATH_T, PATH_N, and PATH_M based on the pathological TNM criteria. The TNM staging system is crucial for assessing trends in cancer treatment and control. It is used by physicians for prognosis estimation, treatment planning, evaluation of therapies, outcome analysis, follow-up planning, and assessing early detection results. Pathological staging is primarily determined by pathologists, with input from other medical specialists. It is usually written in the format of T(PATH_T code)N(PATH_N code)[M(PATH_M code)], where the PATH_M code may be omitted. For example, in the description 'T1b1N2cM0', the PATH_T, PATH_N, and PATH_M codes are 1b, 2c, and 0, respectively. In cases where no explicit description is provided by physicians, you should make informed judgments based on available information in the medical records.
# Here is the detail definition for the three codes.
# - PATH_T: This code indicates the size or extent of the primary tumor.
# -- Available codes: X, 0, A, IS, ISU, ISD, ISDC, ISPA, ISLA, 1M, 2, 2A, 2A1, 2A2, 2B, 2C, 2D, 3, 3A, 3B, 1, 1A, 1A1, 1A2, 1B, 1B1, 1B2, 1B3, 1C, 1C1, 1C2, 1C3, 1D, 3C, 3D, 4, 4A, 4B, 4C, 4D, 4E, 8888 (Not applicable), 9999 (Not recorded)
# - PATH_N: This code represents the presence and extent of regional lymph node metastasis.
# -- Available codes: X, 0, 0A, 0B, 0C, 0D, 1, 1A, 1AS, 1B, 1C, 1M, 2M, 2, 2A, 2B, 2C, 3, 3A, 3B, 3C, 888 (Not applicable), 999 (Not recorded)
# - PATH_M: This code indicates the presence of distant metastasis.
# -- Available codes: 1, 1A, 1A0, 1A1, 1B, 1B0, 1B1, 1C, 1C0, 1C1, 1D, 1D0, C, CA, CA0, CA1, CB, CB0, CB1, CC, CC0, CC1, CD, CD0, CD1, CE, 888 (Not applicable), 999 (Not recorded)

# Please ensure your response is limited to one of these specific codes. For example, if the report states 'pT3N0M2b', your coding results in this case should be:
# PATH_T: 3
# PATH_N: 0
# PATH_M: 2B
# After providing the code, please add two newline characters and then explain the reason for your classification choice.'''
# # Here are some examples to guide you:
# # Example 1: The pathology report states 'T1a1N1'. Your coding results in this case should be:
# # PATH_T: 1A1
# # PATH_N: 1

# # Example 2: The pathology report states 'pT3N0M2b'. Your coding results in this case should be:
# # Results:
# # PATH_T: 3
# # PATH_N: 0
# # PATH_M: 2B

# # After providing the codes, add two newline characters and then elaborate on the reasoning behind your coding results. This explanation should clearly articulate how the information from the medical records or pathology reports influenced your coding decisions.'''

# llama_templates = [#{'role': 'user', 'content': llama_template_laterality}, {'role': 'user', 'content': llama_template_primary_site},
#                   #  {'role': 'user', 'content': llama_template_path_TNM},
#                 #    {'role': 'user', 'content': llama_template_behavior}, {'role': 'user', 'content': llama_template_node_examine},
#                 #    {'role': 'user', 'content': llama_template_node_positive},
#                     {'role': 'user', 'content': llama_template_histology},
#                     {'role': 'user', 'content': llama_template_primary_site_laterality}]

agent_document_idx_templates = {
    "histology": llama_template_histology,
    "primary_site_and_laterality": llama_template_primary_site_laterality
}

def update_query_engine_prompts(tool_prompt, query_engine):
    qa_prompt_tmpl_str = """Context information is below.
---------------------
{context_str}
---------------------
Given the context information and the following knowledge and command, answer the query.
Knowledge and command: {cancer_registry_kg}
Query: {query_str}
Answer: \
"""
    prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)
    partial_prompt_tmpl = prompt_tmpl.partial_format(cancer_registry_kg=tool_prompt)
    print(partial_prompt_tmpl)
    query_engine.update_prompts({"response_synthesizer:text_qa_template": partial_prompt_tmpl})


In [ ]:
"""ReAct agent worker."""

import asyncio
import uuid
from itertools import chain
from threading import Thread
from typing import (
    Any,
    AsyncGenerator,
    Dict,
    Generator,
    List,
    Optional,
    Sequence,
    Tuple,
    cast,
)

from llama_index.agent.react.formatter import ReActChatFormatter
from llama_index.agent.react.output_parser import ReActOutputParser
from llama_index.agent.react.types import (
    ActionReasoningStep,
    BaseReasoningStep,
    ObservationReasoningStep,
    ResponseReasoningStep,
)
from llama_index.agent.types import (
    BaseAgentWorker,
    Task,
    TaskStep,
    TaskStepOutput,
)
from llama_index.callbacks import (
    CallbackManager,
    CBEventType,
    EventPayload,
    trace_method,
)
from llama_index.chat_engine.types import (
    AGENT_CHAT_RESPONSE_TYPE,
    AgentChatResponse,
    StreamingAgentChatResponse,
)
from llama_index.llms.base import ChatMessage, ChatResponse
from llama_index.llms.llm import LLM
from llama_index.llms.openai import OpenAI
from llama_index.llms.types import MessageRole
from llama_index.memory.chat_memory_buffer import ChatMemoryBuffer
from llama_index.memory.types import BaseMemory
from llama_index.objects.base import ObjectRetriever
from llama_index.tools import BaseTool, ToolOutput, adapt_to_async_tool
from llama_index.tools.types import AsyncBaseTool
from llama_index.utils import print_text, unit_generator

DEFAULT_MODEL_NAME = "gpt-3.5-turbo-0613"


def add_user_step_to_reasoning(
    step: TaskStep,
    memory: BaseMemory,
    current_reasoning: List[BaseReasoningStep],
    verbose: bool = False,
) -> None:
    """Add user step to memory."""
    if "is_first" in step.step_state and step.step_state["is_first"]:
        # add to new memory
        memory.put(ChatMessage(content=step.input, role=MessageRole.USER))
        step.step_state["is_first"] = False
    else:
        reasoning_step = ObservationReasoningStep(observation=step.input)
        current_reasoning.append(reasoning_step)
        if verbose:
            print(f"Added user message to memory: {step.input}")


class ReActAgentWorker(BaseAgentWorker):
    """OpenAI Agent worker."""

    def __init__(
        self,
        tools: Sequence[BaseTool],
        llm: LLM,
        max_iterations: int = 10,
        react_chat_formatter: Optional[ReActChatFormatter] = None,
        output_parser: Optional[ReActOutputParser] = None,
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = False,
        tool_retriever: Optional[ObjectRetriever[BaseTool]] = None,
    ) -> None:
        self._llm = llm
        self.callback_manager = callback_manager or llm.callback_manager
        self._max_iterations = max_iterations
        self._react_chat_formatter = react_chat_formatter or ReActChatFormatter()
        self._output_parser = output_parser or ReActOutputParser()
        self._verbose = verbose

        if len(tools) > 0 and tool_retriever is not None:
            raise ValueError("Cannot specify both tools and tool_retriever")
        elif len(tools) > 0:
            self._get_tools = lambda _: tools
        elif tool_retriever is not None:
            tool_retriever_c = cast(ObjectRetriever[BaseTool], tool_retriever)
            self._get_tools = lambda message: tool_retriever_c.retrieve(message)
        else:
            self._get_tools = lambda _: []

    @classmethod
    def from_tools(
        cls,
        tools: Optional[Sequence[BaseTool]] = None,
        tool_retriever: Optional[ObjectRetriever[BaseTool]] = None,
        llm: Optional[LLM] = None,
        max_iterations: int = 10,
        react_chat_formatter: Optional[ReActChatFormatter] = None,
        output_parser: Optional[ReActOutputParser] = None,
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = False,
        **kwargs: Any,
    ) -> "ReActAgentWorker":
        """Convenience constructor method from set of of BaseTools (Optional).

        NOTE: kwargs should have been exhausted by this point. In other words
        the various upstream components such as BaseSynthesizer (response synthesizer)
        or BaseRetriever should have picked up off their respective kwargs in their
        constructions.

        Returns:
            ReActAgent
        """
        llm = llm or OpenAI(model=DEFAULT_MODEL_NAME)
        if callback_manager is not None:
            llm.callback_manager = callback_manager
        return cls(
            tools=tools or [],
            tool_retriever=tool_retriever,
            llm=llm,
            max_iterations=max_iterations,
            react_chat_formatter=react_chat_formatter,
            output_parser=output_parser,
            callback_manager=callback_manager,
            verbose=verbose,
        )

    def initialize_step(self, task: Task, **kwargs: Any) -> TaskStep:
        """Initialize step from task."""
        sources: List[ToolOutput] = []
        current_reasoning: List[BaseReasoningStep] = []
        # temporary memory for new messages
        new_memory = ChatMemoryBuffer.from_defaults()

        # initialize task state
        task_state = {
            "sources": sources,
            "current_reasoning": current_reasoning,
            "new_memory": new_memory,
        }
        task.extra_state.update(task_state)

        return TaskStep(
            task_id=task.task_id,
            step_id=str(uuid.uuid4()),
            input=task.input,
            step_state={"is_first": True},
        )

    def get_tools(self, input: str) -> List[AsyncBaseTool]:
        """Get tools."""
        return [adapt_to_async_tool(t) for t in self._get_tools(input)]

    def _extract_reasoning_step(
        self, output: ChatResponse, is_streaming: bool = False
    ) -> Tuple[str, List[BaseReasoningStep], bool]:
        """
        Extracts the reasoning step from the given output.

        This method parses the message content from the output,
        extracts the reasoning step, and determines whether the processing is
        complete. It also performs validation checks on the output and
        handles possible errors.
        """
        if output.message.content is None:
            raise ValueError("Got empty message.")
        message_content = output.message.content
        current_reasoning = []
        try:
            reasoning_step = self._output_parser.parse(message_content, is_streaming)
        except BaseException as exc:
            raise ValueError(f"Could not parse output: {message_content}") from exc
        if self._verbose:
            print_text(f"{reasoning_step.get_content()}\n", color="pink")
        current_reasoning.append(reasoning_step)

        if reasoning_step.is_done:
            return message_content, current_reasoning, True

        reasoning_step = cast(ActionReasoningStep, reasoning_step)
        if not isinstance(reasoning_step, ActionReasoningStep):
            raise ValueError(f"Expected ActionReasoningStep, got {reasoning_step}")

        return message_content, current_reasoning, False

    def _process_actions(
        self,
        task: Task,
        tools: Sequence[AsyncBaseTool],
        output: ChatResponse,
        is_streaming: bool = False,
    ) -> Tuple[List[BaseReasoningStep], bool]:
        tools_dict: Dict[str, AsyncBaseTool] = {
            tool.metadata.get_name(): tool for tool in tools
        }
        _, current_reasoning, is_done = self._extract_reasoning_step(
            output, is_streaming
        )

        if is_done:
            return current_reasoning, True

        # call tool with input
        reasoning_step = cast(ActionReasoningStep, current_reasoning[-1])
        tool = tools_dict[reasoning_step.action]
        with self.callback_manager.event(
            CBEventType.FUNCTION_CALL,
            payload={
                EventPayload.FUNCTION_CALL: reasoning_step.action_input,
                EventPayload.TOOL: tool.metadata,
            },
        ) as event:
            tool_output = tool.call(**reasoning_step.action_input)
            event.on_end(payload={EventPayload.FUNCTION_OUTPUT: str(tool_output)})

        task.extra_state["sources"].append(tool_output)

        observation_step = ObservationReasoningStep(observation=str(tool_output))
        current_reasoning.append(observation_step)
        if self._verbose:
            print_text(f"{observation_step.get_content()}\n", color="blue")
        return current_reasoning, False

    async def _aprocess_actions(
        self,
        task: Task,
        tools: Sequence[AsyncBaseTool],
        output: ChatResponse,
        is_streaming: bool = False,
    ) -> Tuple[List[BaseReasoningStep], bool]:
        tools_dict = {tool.metadata.name: tool for tool in tools}
        _, current_reasoning, is_done = self._extract_reasoning_step(
            output, is_streaming
        )

        if is_done:
            return current_reasoning, True

        # call tool with input
        reasoning_step = cast(ActionReasoningStep, current_reasoning[-1])
        tool = tools_dict[reasoning_step.action]
        with self.callback_manager.event(
            CBEventType.FUNCTION_CALL,
            payload={
                EventPayload.FUNCTION_CALL: reasoning_step.action_input,
                EventPayload.TOOL: tool.metadata,
            },
        ) as event:
            tool_output = await tool.acall(**reasoning_step.action_input)
            event.on_end(payload={EventPayload.FUNCTION_OUTPUT: str(tool_output)})

        task.extra_state["sources"].append(tool_output)

        observation_step = ObservationReasoningStep(observation=str(tool_output))
        current_reasoning.append(observation_step)
        if self._verbose:
            print_text(f"{observation_step.get_content()}\n", color="blue")
        return current_reasoning, False

    def _get_response(
        self,
        current_reasoning: List[BaseReasoningStep],
        sources: List[ToolOutput],
    ) -> AgentChatResponse:
        """Get response from reasoning steps."""
        if len(current_reasoning) == 0:
            raise ValueError("No reasoning steps were taken.")
        elif len(current_reasoning) == self._max_iterations:
            raise ValueError("Reached max iterations.")

        if isinstance(current_reasoning[-1], ResponseReasoningStep):
            response_step = cast(ResponseReasoningStep, current_reasoning[-1])
            response_str = response_step.response
        else:
            response_str = current_reasoning[-1].get_content()

        # TODO: add sources from reasoning steps
        return AgentChatResponse(response=response_str, sources=sources)

    def _get_task_step_response(
        self, agent_response: AGENT_CHAT_RESPONSE_TYPE, step: TaskStep, is_done: bool
    ) -> TaskStepOutput:
        """Get task step response."""
        if is_done:
            new_steps = []
        else:
            new_steps = [
                step.get_next_step(
                    step_id=str(uuid.uuid4()),
                    # NOTE: input is unused
                    input=None,
                )
            ]

        return TaskStepOutput(
            output=agent_response,
            task_step=step,
            is_last=is_done,
            next_steps=new_steps,
        )

    def _infer_stream_chunk_is_final(self, chunk: ChatResponse) -> bool:
        """Infers if a chunk from a live stream is the start of the final
        reasoning step. (i.e., and should eventually become
        ResponseReasoningStep — not part of this function's logic tho.).

        Args:
            chunk (ChatResponse): the current chunk stream to check

        Returns:
            bool: Boolean on whether the chunk is the start of the final response
        """
        latest_content = chunk.message.content
        if latest_content:
            if not latest_content.startswith(
                "Thought"
            ):  # doesn't follow thought-action format
                return True
            else:
                if "Answer: " in latest_content:
                    return True
        return False

    def _add_back_chunk_to_stream(
        self, chunk: ChatResponse, chat_stream: Generator[ChatResponse, None, None]
    ) -> Generator[ChatResponse, None, None]:
        """Helper method for adding back initial chunk stream of final response
        back to the rest of the chat_stream.

        Args:
            chunk (ChatResponse): the chunk to add back to the beginning of the
                                    chat_stream.

        Return:
            Generator[ChatResponse, None, None]: the updated chat_stream
        """
        updated_stream = chain.from_iterable(  # need to add back partial response chunk
            [
                unit_generator(chunk),
                chat_stream,
            ]
        )
        # use cast to avoid mypy issue with chain and Generator
        updated_stream_c: Generator[ChatResponse, None, None] = cast(
            Generator[ChatResponse, None, None], updated_stream
        )
        return updated_stream_c

    async def _async_add_back_chunk_to_stream(
        self, chunk: ChatResponse, chat_stream: AsyncGenerator[ChatResponse, None]
    ) -> AsyncGenerator[ChatResponse, None]:
        """Helper method for adding back initial chunk stream of final response
        back to the rest of the chat_stream.

        NOTE: this itself is not an async function.

        Args:
            chunk (ChatResponse): the chunk to add back to the beginning of the
                                    chat_stream.

        Return:
            AsyncGenerator[ChatResponse, None]: the updated async chat_stream
        """
        yield chunk
        async for item in chat_stream:
            yield item

    def _run_step(
        self,
        step: TaskStep,
        task: Task,
    ) -> TaskStepOutput:
        """Run step."""
        if step.input is not None:
            add_user_step_to_reasoning(
                step,
                task.extra_state["new_memory"],
                task.extra_state["current_reasoning"],
                verbose=self._verbose,
            )
        # TODO: see if we want to do step-based inputs
        tools = self.get_tools(task.input)

        input_chat = self._react_chat_formatter.format(
            tools,
            chat_history=task.memory.get() + task.extra_state["new_memory"].get_all(),
            current_reasoning=task.extra_state["current_reasoning"],
        )

        # send prompt
        chat_response = self._llm.chat(input_chat)
        # given react prompt outputs, call tools or return response
        reasoning_steps, is_done = self._process_actions(
            task, tools, output=chat_response
        )
        task.extra_state["current_reasoning"].extend(reasoning_steps)
        agent_response = self._get_response(
            task.extra_state["current_reasoning"], task.extra_state["sources"]
        )
        if is_done:
            task.extra_state["new_memory"].put(
                ChatMessage(content=agent_response.response, role=MessageRole.ASSISTANT)
            )

        return self._get_task_step_response(agent_response, step, is_done)

    async def _arun_step(
        self,
        step: TaskStep,
        task: Task,
    ) -> TaskStepOutput:
        """Run step."""
        if step.input is not None:
            add_user_step_to_reasoning(
                step,
                task.extra_state["new_memory"],
                task.extra_state["current_reasoning"],
                verbose=self._verbose,
            )
        # TODO: see if we want to do step-based inputs
        tools = self.get_tools(task.input)

        input_chat = self._react_chat_formatter.format(
            tools,
            chat_history=task.memory.get() + task.extra_state["new_memory"].get_all(),
            current_reasoning=task.extra_state["current_reasoning"],
        )
        # send prompt
        chat_response = await self._llm.achat(input_chat)
        # given react prompt outputs, call tools or return response
        reasoning_steps, is_done = await self._aprocess_actions(
            task, tools, output=chat_response
        )
        task.extra_state["current_reasoning"].extend(reasoning_steps)
        agent_response = self._get_response(
            task.extra_state["current_reasoning"], task.extra_state["sources"]
        )
        if is_done:
            task.extra_state["new_memory"].put(
                ChatMessage(content=agent_response.response, role=MessageRole.ASSISTANT)
            )

        return self._get_task_step_response(agent_response, step, is_done)

    def _run_step_stream(
        self,
        step: TaskStep,
        task: Task,
    ) -> TaskStepOutput:
        """Run step."""
        if step.input is not None:
            add_user_step_to_reasoning(
                step,
                task.extra_state["new_memory"],
                task.extra_state["current_reasoning"],
                verbose=self._verbose,
            )
        # TODO: see if we want to do step-based inputs
        tools = self.get_tools(task.input)

        input_chat = self._react_chat_formatter.format(
            tools,
            chat_history=task.memory.get() + task.extra_state["new_memory"].get_all(),
            current_reasoning=task.extra_state["current_reasoning"],
        )

        chat_stream = self._llm.stream_chat(input_chat)

        # iterate over stream, break out if is final answer after the "Answer: "
        full_response = ChatResponse(
            message=ChatMessage(content=None, role="assistant")
        )
        is_done = False
        for latest_chunk in chat_stream:
            full_response = latest_chunk
            is_done = self._infer_stream_chunk_is_final(latest_chunk)
            if is_done:
                break

        if not is_done:
            # given react prompt outputs, call tools or return response
            reasoning_steps, _ = self._process_actions(
                task, tools=tools, output=full_response, is_streaming=True
            )
            task.extra_state["current_reasoning"].extend(reasoning_steps)
            # use _get_response to return intermediate response
            agent_response: AGENT_CHAT_RESPONSE_TYPE = self._get_response(
                task.extra_state["current_reasoning"], task.extra_state["sources"]
            )
        else:
            # Get the response in a separate thread so we can yield the response
            response_stream = self._add_back_chunk_to_stream(
                chunk=latest_chunk, chat_stream=chat_stream
            )

            agent_response = StreamingAgentChatResponse(
                chat_stream=response_stream,
                sources=task.extra_state["sources"],
            )
            thread = Thread(
                target=agent_response.write_response_to_history,
                args=(task.extra_state["new_memory"],),
            )
            thread.start()

        return self._get_task_step_response(agent_response, step, is_done)

    async def _arun_step_stream(
        self,
        step: TaskStep,
        task: Task,
    ) -> TaskStepOutput:
        """Run step."""
        if step.input is not None:
            add_user_step_to_reasoning(
                step,
                task.extra_state["new_memory"],
                task.extra_state["current_reasoning"],
                verbose=self._verbose,
            )
        # TODO: see if we want to do step-based inputs
        tools = self.get_tools(task.input)

        input_chat = self._react_chat_formatter.format(
            tools,
            chat_history=task.memory.get() + task.extra_state["new_memory"].get_all(),
            current_reasoning=task.extra_state["current_reasoning"],
        )

        chat_stream = await self._llm.astream_chat(input_chat)

        # iterate over stream, break out if is final answer after the "Answer: "
        full_response = ChatResponse(
            message=ChatMessage(content=None, role="assistant")
        )
        is_done = False
        async for latest_chunk in chat_stream:
            full_response = latest_chunk
            is_done = self._infer_stream_chunk_is_final(latest_chunk)
            if is_done:
                break

        if not is_done:
            # given react prompt outputs, call tools or return response
            reasoning_steps, _ = self._process_actions(
                task, tools=tools, output=full_response, is_streaming=True
            )
            task.extra_state["current_reasoning"].extend(reasoning_steps)
            # use _get_response to return intermediate response
            agent_response: AGENT_CHAT_RESPONSE_TYPE = self._get_response(
                task.extra_state["current_reasoning"], task.extra_state["sources"]
            )
        else:
            # Get the response in a separate thread so we can yield the response
            response_stream = self._async_add_back_chunk_to_stream(
                chunk=latest_chunk, chat_stream=chat_stream
            )

            agent_response = StreamingAgentChatResponse(
                achat_stream=response_stream,
                sources=task.extra_state["sources"],
            )
            # create task to write chat response to history
            asyncio.create_task(
                agent_response.awrite_response_to_history(
                    task.extra_state["new_memory"]
                )
            )
            # wait until response writing is done
            await agent_response._is_function_false_event.wait()

        return self._get_task_step_response(agent_response, step, is_done)

    @trace_method("run_step")
    def run_step(self, step: TaskStep, task: Task, **kwargs: Any) -> TaskStepOutput:
        """Run step."""
        return self._run_step(step, task)

    @trace_method("run_step")
    async def arun_step(
        self, step: TaskStep, task: Task, **kwargs: Any
    ) -> TaskStepOutput:
        """Run step (async)."""
        return await self._arun_step(step, task)

    @trace_method("run_step")
    def stream_step(self, step: TaskStep, task: Task, **kwargs: Any) -> TaskStepOutput:
        """Run step (stream)."""
        # TODO: figure out if we need a different type for TaskStepOutput
        return self._run_step_stream(step, task)

    @trace_method("run_step")
    async def astream_step(
        self, step: TaskStep, task: Task, **kwargs: Any
    ) -> TaskStepOutput:
        """Run step (async stream)."""
        return await self._arun_step_stream(step, task)

    def finalize_task(self, task: Task, **kwargs: Any) -> None:
        """Finalize task, after all the steps are completed."""
        # add new messages to memory
        task.memory.set(task.memory.get() + task.extra_state["new_memory"].get_all())
        # reset new memory
        task.extra_state["new_memory"].reset()
        
import uuid
from abc import abstractmethod
from collections import deque
from typing import Any, Deque, Dict, List, Optional, Union, cast

from llama_index.agent.types import (
    BaseAgent,
    BaseAgentWorker,
    Task,
    TaskStep,
    TaskStepOutput,
)
from llama_index.bridge.pydantic import BaseModel, Field
from llama_index.callbacks import (
    CallbackManager,
    CBEventType,
    EventPayload,
    trace_method,
)
from llama_index.chat_engine.types import (
    AGENT_CHAT_RESPONSE_TYPE,
    AgentChatResponse,
    ChatResponseMode,
    StreamingAgentChatResponse,
)
from llama_index.llms.base import ChatMessage
from llama_index.llms.llm import LLM
from llama_index.memory import BaseMemory, ChatMemoryBuffer
from llama_index.memory.types import BaseMemory


class BaseAgentRunner(BaseAgent):
    """Base agent runner."""

    @abstractmethod
    def create_task(self, input: str, **kwargs: Any) -> Task:
        """Create task."""

    @abstractmethod
    def delete_task(
        self,
        task_id: str,
    ) -> None:
        """Delete task.

        NOTE: this will not delete any previous executions from memory.

        """

    @abstractmethod
    def list_tasks(self, **kwargs: Any) -> List[Task]:
        """List tasks."""

    @abstractmethod
    def get_task(self, task_id: str, **kwargs: Any) -> Task:
        """Get task."""

    @abstractmethod
    def get_upcoming_steps(self, task_id: str, **kwargs: Any) -> List[TaskStep]:
        """Get upcoming steps."""

    @abstractmethod
    def get_completed_steps(self, task_id: str, **kwargs: Any) -> List[TaskStepOutput]:
        """Get completed steps."""

    def get_completed_step(
        self, task_id: str, step_id: str, **kwargs: Any
    ) -> TaskStepOutput:
        """Get completed step."""
        # call get_completed_steps, and then find the right task
        completed_steps = self.get_completed_steps(task_id, **kwargs)
        for step_output in completed_steps:
            if step_output.task_step.step_id == step_id:
                return step_output
        raise ValueError(f"Could not find step_id: {step_id}")

    @abstractmethod
    def run_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step."""

    @abstractmethod
    async def arun_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (async)."""

    @abstractmethod
    def stream_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (stream)."""

    @abstractmethod
    async def astream_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (async stream)."""

    @abstractmethod
    def finalize_response(
        self,
        task_id: str,
        step_output: Optional[TaskStepOutput] = None,
    ) -> AGENT_CHAT_RESPONSE_TYPE:
        """Finalize response."""

    @abstractmethod
    def undo_step(self, task_id: str) -> None:
        """Undo previous step."""
        raise NotImplementedError("undo_step not implemented")


def validate_step_from_args(
    task_id: str, input: Optional[str] = None, step: Optional[Any] = None, **kwargs: Any
) -> Optional[TaskStep]:
    """Validate step from args."""
    if step is not None:
        if input is not None:
            raise ValueError("Cannot specify both `step` and `input`")
        if not isinstance(step, TaskStep):
            raise ValueError(f"step must be TaskStep: {step}")
        return step
    else:
        return (
            None
            if input is None
            else TaskStep(
                task_id=task_id, step_id=str(uuid.uuid4()), input=input, **kwargs
            )
        )


class TaskState(BaseModel):
    """Task state."""

    task: Task = Field(..., description="Task.")
    step_queue: Deque[TaskStep] = Field(
        default_factory=deque, description="Task step queue."
    )
    completed_steps: List[TaskStepOutput] = Field(
        default_factory=list, description="Completed step outputs."
    )


class AgentState(BaseModel):
    """Agent state."""

    task_dict: Dict[str, TaskState] = Field(
        default_factory=dict, description="Task dictionary."
    )

    def get_task(self, task_id: str) -> Task:
        """Get task state."""
        return self.task_dict[task_id].task

    def get_completed_steps(self, task_id: str) -> List[TaskStepOutput]:
        """Get completed steps."""
        return self.task_dict[task_id].completed_steps

    def get_step_queue(self, task_id: str) -> Deque[TaskStep]:
        """Get step queue."""
        return self.task_dict[task_id].step_queue


class AgentRunner(BaseAgentRunner):
    """Agent runner.

    Top-level agent orchestrator that can create tasks, run each step in a task,
    or run a task e2e. Stores state and keeps track of tasks.

    Args:
        agent_worker (BaseAgentWorker): step executor
        chat_history (Optional[List[ChatMessage]], optional): chat history. Defaults to None.
        state (Optional[AgentState], optional): agent state. Defaults to None.
        memory (Optional[BaseMemory], optional): memory. Defaults to None.
        llm (Optional[LLM], optional): LLM. Defaults to None.
        callback_manager (Optional[CallbackManager], optional): callback manager. Defaults to None.
        init_task_state_kwargs (Optional[dict], optional): init task state kwargs. Defaults to None.

    """

    # # TODO: implement this in Pydantic

    def __init__(
        self,
        agent_worker: BaseAgentWorker,
        chat_history: Optional[List[ChatMessage]] = None,
        state: Optional[AgentState] = None,
        memory: Optional[BaseMemory] = None,
        llm: Optional[LLM] = None,
        callback_manager: Optional[CallbackManager] = None,
        init_task_state_kwargs: Optional[dict] = None,
        delete_task_on_finish: bool = False,
    ) -> None:
        """Initialize."""
        self.agent_worker = agent_worker
        self.state = state or AgentState()
        self.memory = memory or ChatMemoryBuffer.from_defaults(chat_history, llm=llm)
        self.callback_manager = callback_manager or CallbackManager([])
        self.init_task_state_kwargs = init_task_state_kwargs or {}
        self.delete_task_on_finish = delete_task_on_finish

    @property
    def chat_history(self) -> List[ChatMessage]:
        return self.memory.get_all()

    def reset(self) -> None:
        self.memory.reset()

    def create_task(self, input: str, **kwargs: Any) -> Task:
        """Create task."""
        task = Task(
            input=input,
            memory=self.memory,
            extra_state=self.init_task_state_kwargs,
            **kwargs,
        )
        # # put input into memory
        # self.memory.put(ChatMessage(content=input, role=MessageRole.USER))

        # get initial step from task, and put it in the step queue
        initial_step = self.agent_worker.initialize_step(task)
        task_state = TaskState(
            task=task,
            step_queue=deque([initial_step]),
        )
        # add it to state
        self.state.task_dict[task.task_id] = task_state

        return task

    def delete_task(
        self,
        task_id: str,
    ) -> None:
        """Delete task.

        NOTE: this will not delete any previous executions from memory.

        """
        self.state.task_dict.pop(task_id)

    def list_tasks(self, **kwargs: Any) -> List[Task]:
        """List tasks."""
        return list(self.state.task_dict.values())

    def get_task(self, task_id: str, **kwargs: Any) -> Task:
        """Get task."""
        return self.state.get_task(task_id)

    def get_upcoming_steps(self, task_id: str, **kwargs: Any) -> List[TaskStep]:
        """Get upcoming steps."""
        return list(self.state.get_step_queue(task_id))

    def get_completed_steps(self, task_id: str, **kwargs: Any) -> List[TaskStepOutput]:
        """Get completed steps."""
        return self.state.get_completed_steps(task_id)

    def _run_step(
        self,
        task_id: str,
        step: Optional[TaskStep] = None,
        mode: ChatResponseMode = ChatResponseMode.WAIT,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Execute step."""
        task = self.state.get_task(task_id)
        step_queue = self.state.get_step_queue(task_id)
        step = step or step_queue.popleft()

        # TODO: figure out if you can dynamically swap in different step executors
        # not clear when you would do that by theoretically possible

        if mode == ChatResponseMode.WAIT:
            cur_step_output = self.agent_worker.run_step(step, task, **kwargs)
        elif mode == ChatResponseMode.STREAM:
            cur_step_output = self.agent_worker.stream_step(step, task, **kwargs)
        else:
            raise ValueError(f"Invalid mode: {mode}")
        # append cur_step_output next steps to queue
        next_steps = cur_step_output.next_steps
        step_queue.extend(next_steps)

        # add cur_step_output to completed steps
        completed_steps = self.state.get_completed_steps(task_id)
        completed_steps.append(cur_step_output)

        return cur_step_output

    async def _arun_step(
        self,
        task_id: str,
        step: Optional[TaskStep] = None,
        mode: ChatResponseMode = ChatResponseMode.WAIT,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Execute step."""
        task = self.state.get_task(task_id)
        step_queue = self.state.get_step_queue(task_id)
        step = step or step_queue.popleft()

        # TODO: figure out if you can dynamically swap in different step executors
        # not clear when you would do that by theoretically possible
        if mode == ChatResponseMode.WAIT:
            cur_step_output = await self.agent_worker.arun_step(step, task, **kwargs)
        elif mode == ChatResponseMode.STREAM:
            cur_step_output = await self.agent_worker.astream_step(step, task, **kwargs)
        else:
            raise ValueError(f"Invalid mode: {mode}")
        # append cur_step_output next steps to queue
        next_steps = cur_step_output.next_steps
        step_queue.extend(next_steps)

        # add cur_step_output to completed steps
        completed_steps = self.state.get_completed_steps(task_id)
        completed_steps.append(cur_step_output)

        return cur_step_output

    def run_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step."""
        step = validate_step_from_args(task_id, input, step, **kwargs)
        return self._run_step(task_id, step, mode=ChatResponseMode.WAIT, **kwargs)

    async def arun_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (async)."""
        step = validate_step_from_args(task_id, input, step, **kwargs)
        return await self._arun_step(
            task_id, step, mode=ChatResponseMode.WAIT, **kwargs
        )

    def stream_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (stream)."""
        step = validate_step_from_args(task_id, input, step, **kwargs)
        return self._run_step(task_id, step, mode=ChatResponseMode.STREAM, **kwargs)

    async def astream_step(
        self,
        task_id: str,
        input: Optional[str] = None,
        step: Optional[TaskStep] = None,
        **kwargs: Any,
    ) -> TaskStepOutput:
        """Run step (async stream)."""
        step = validate_step_from_args(task_id, input, step, **kwargs)
        return await self._arun_step(
            task_id, step, mode=ChatResponseMode.STREAM, **kwargs
        )

    def finalize_response(
        self,
        task_id: str,
        step_output: Optional[TaskStepOutput] = None,
    ) -> AGENT_CHAT_RESPONSE_TYPE:
        """Finalize response."""
        if step_output is None:
            step_output = self.state.get_completed_steps(task_id)[-1]
        if not step_output.is_last:
            raise ValueError(
                "finalize_response can only be called on the last step output"
            )

        if not isinstance(
            step_output.output,
            (AgentChatResponse, StreamingAgentChatResponse),
        ):
            raise ValueError(
                "When `is_last` is True, cur_step_output.output must be "
                f"AGENT_CHAT_RESPONSE_TYPE: {step_output.output}"
            )

        # finalize task
        self.agent_worker.finalize_task(self.state.get_task(task_id))

        if self.delete_task_on_finish:
            self.delete_task(task_id)

        return cast(AGENT_CHAT_RESPONSE_TYPE, step_output.output)

    def _chat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
        mode: ChatResponseMode = ChatResponseMode.WAIT,
    ) -> AGENT_CHAT_RESPONSE_TYPE:
        """Chat with step executor."""
        if chat_history is not None:
            self.memory.set(chat_history)
        task = self.create_task(message)

        result_output = None
        while True:
            # pass step queue in as argument, assume step executor is stateless
            cur_step_output = self._run_step(task.task_id, mode=mode)

            if cur_step_output.is_last:
                result_output = cur_step_output
                break

        return self.finalize_response(task.task_id, result_output)

    async def _achat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
        mode: ChatResponseMode = ChatResponseMode.WAIT,
    ) -> AGENT_CHAT_RESPONSE_TYPE:
        """Chat with step executor."""
        if chat_history is not None:
            self.memory.set(chat_history)
        task = self.create_task(message)

        result_output = None
        while True:
            # pass step queue in as argument, assume step executor is stateless
            cur_step_output = await self._arun_step(task.task_id, mode=mode)

            if cur_step_output.is_last:
                result_output = cur_step_output
                break

        return self.finalize_response(task.task_id, result_output)

    @trace_method("chat")
    def chat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
    ) -> AgentChatResponse:
        with self.callback_manager.event(
            CBEventType.AGENT_STEP,
            payload={EventPayload.MESSAGES: [message]},
        ) as e:
            chat_response = self._chat(
                message, chat_history, tool_choice, mode=ChatResponseMode.WAIT
            )
            assert isinstance(chat_response, AgentChatResponse)
            e.on_end(payload={EventPayload.RESPONSE: chat_response})
        return chat_response

    @trace_method("chat")
    async def achat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
    ) -> AgentChatResponse:
        with self.callback_manager.event(
            CBEventType.AGENT_STEP,
            payload={EventPayload.MESSAGES: [message]},
        ) as e:
            chat_response = await self._achat(
                message, chat_history, tool_choice, mode=ChatResponseMode.WAIT
            )
            assert isinstance(chat_response, AgentChatResponse)
            e.on_end(payload={EventPayload.RESPONSE: chat_response})
        return chat_response

    @trace_method("chat")
    def stream_chat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
    ) -> StreamingAgentChatResponse:
        with self.callback_manager.event(
            CBEventType.AGENT_STEP,
            payload={EventPayload.MESSAGES: [message]},
        ) as e:
            chat_response = self._chat(
                message, chat_history, tool_choice, mode=ChatResponseMode.STREAM
            )
            assert isinstance(chat_response, StreamingAgentChatResponse)
            e.on_end(payload={EventPayload.RESPONSE: chat_response})
        return chat_response

    @trace_method("chat")
    async def astream_chat(
        self,
        message: str,
        chat_history: Optional[List[ChatMessage]] = None,
        tool_choice: Union[str, dict] = "auto",
    ) -> StreamingAgentChatResponse:
        with self.callback_manager.event(
            CBEventType.AGENT_STEP,
            payload={EventPayload.MESSAGES: [message]},
        ) as e:
            chat_response = await self._achat(
                message, chat_history, tool_choice, mode=ChatResponseMode.STREAM
            )
            assert isinstance(chat_response, StreamingAgentChatResponse)
            e.on_end(payload={EventPayload.RESPONSE: chat_response})
        return chat_response

    def undo_step(self, task_id: str) -> None:
        """Undo previous step."""
        raise NotImplementedError("undo_step not implemented")

"""ReAct agent.

Simple wrapper around AgentRunner + ReActAgentWorker.

For the legacy implementation see:
```python
from llama_index.agent.legacy.react.base import ReActAgent
```

"""
from typing import (
    Any,
    List,
    Optional,
    Sequence,
    Type,
)

from llama_index.agent.react.formatter import ReActChatFormatter
from llama_index.agent.react.output_parser import ReActOutputParser
#from llama_index.agent.react.step import ReActAgentWorker
#from llama_index.agent.runner.base import AgentRunner
from llama_index.callbacks import (
    CallbackManager,
)
from llama_index.llms.llm import LLM
from llama_index.llms.openai import OpenAI
from llama_index.llms.types import ChatMessage
from llama_index.memory.chat_memory_buffer import ChatMemoryBuffer
from llama_index.memory.types import BaseMemory
from llama_index.objects.base import ObjectRetriever
from llama_index.tools import BaseTool

DEFAULT_MODEL_NAME = "gpt-3.5-turbo-0613"


class ReActAgent(AgentRunner):
    """ReAct agent.

    Subclasses AgentRunner with a ReActAgentWorker.

    For the legacy implementation see:
    ```python
    from llama_index.agent.legacy.react.base import ReActAgent
    ```

    """

    def __init__(
        self,
        tools: Sequence[BaseTool],
        llm: LLM,
        memory: BaseMemory,
        max_iterations: int = 10,
        react_chat_formatter: Optional[ReActChatFormatter] = None,
        output_parser: Optional[ReActOutputParser] = None,
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = False,
        tool_retriever: Optional[ObjectRetriever[BaseTool]] = None,
    ) -> None:
        """Init params."""
        callback_manager = callback_manager or llm.callback_manager
        step_engine = ReActAgentWorker.from_tools(
            tools=tools,
            tool_retriever=tool_retriever,
            llm=llm,
            max_iterations=max_iterations,
            react_chat_formatter=react_chat_formatter,
            output_parser=output_parser,
            callback_manager=callback_manager,
            verbose=verbose,
        )
        super().__init__(
            step_engine,
            memory=memory,
            llm=llm,
            callback_manager=callback_manager,
        )

    @classmethod
    def from_tools(
        cls,
        tools: Optional[List[BaseTool]] = None,
        tool_retriever: Optional[ObjectRetriever[BaseTool]] = None,
        llm: Optional[LLM] = None,
        chat_history: Optional[List[ChatMessage]] = None,
        memory: Optional[BaseMemory] = None,
        memory_cls: Type[BaseMemory] = ChatMemoryBuffer,
        max_iterations: int = 10,
        react_chat_formatter: Optional[ReActChatFormatter] = None,
        output_parser: Optional[ReActOutputParser] = None,
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = False,
        **kwargs: Any,
    ) -> "ReActAgent":
        """Convenience constructor method from set of of BaseTools (Optional).

        NOTE: kwargs should have been exhausted by this point. In other words
        the various upstream components such as BaseSynthesizer (response synthesizer)
        or BaseRetriever should have picked up off their respective kwargs in their
        constructions.

        Returns:
            ReActAgent
        """
        llm = llm or OpenAI(model=DEFAULT_MODEL_NAME)
        if callback_manager is not None:
            llm.callback_manager = callback_manager
        memory = memory or memory_cls.from_defaults(
            chat_history=chat_history or [], llm=llm
        )
        return cls(
            tools=tools or [],
            tool_retriever=tool_retriever,
            llm=llm,
            memory=memory,
            max_iterations=max_iterations,
            react_chat_formatter=react_chat_formatter,
            output_parser=output_parser,
            callback_manager=callback_manager,
            verbose=verbose,
        )
    
    

from llama_index.response.notebook_utils import display_source_node
from IPython.display import Markdown, display

# define prompt viewing function
def display_prompt_dict(prompts_dict):
    for k, p in prompts_dict.items():
        text_md = f"**Prompt Key**: {k}<br>" f"**Text:** <br>"
        display(Markdown(text_md))
        print(p.get_template())
        display(Markdown("<br><br>"))

import os
from llama_index import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index import load_index_from_storage
from llama_index.node_parser import HierarchicalNodeParser
from llama_index.indices.postprocessor import SentenceTransformerRerank
from llama_index.retrievers import AutoMergingRetriever
from llama_index.query_engine import RetrieverQueryEngine
from llama_index.response.notebook_utils import display_response
from llama_index.tools import QueryEngineTool, ToolMetadata
#from llama_index.agent import ReActAgent
from llama_index.tools.utils import create_schema_from_function

rerank = SentenceTransformerRerank(top_n=6, model="BAAI/bge-reranker-large")

# file_2_index = {}
# file_2_codings = {}
# file_2_engine = {}
similarity_top_k = 12
agent_tools = {}
auto_merging_engines = {}
# for idx, doc in enumerate(docs):
    # auto_merging_engines[doc.metadata['date']] = auto_merging_engine
    # prompts_dict = auto_merging_engine.get_prompts()
    #display_prompt_dict(auto_merging_engine.get_prompts())

    # query_engine_tools = []
    # for agent_name in agent_document_idx_templates.keys():
        
    #     # query_engine_tools = [
    #     #     QueryEngineTool(
    #     #         query_engine=auto_merging_engine, 
    #     #         metadata=ToolMetadata(name=f"get_{agent_name}",
    #     #                           description=(f"Useful for answering questions related to {agent_name} based on the report of {doc.metadata["day"]}"),
    #     # ))]
        
    
        
# See https://docs.llamaindex.ai/en/stable/examples/agent/multi_document_agents-v1.html        
def build_agent_per_doc(doc, nodes, auto_merging_context, agent_document_idx_templates):
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)        
    vi_out_path = f"./data/llamaindex_docs/{file_id}"

    class ToolSchema(BaseModel):
        pass
    #summary_out_path = f"./data/llamaindex_docs/{file_id}_summary.pkl"
    if not os.path.exists(vi_out_path):
        leaf_nodes = get_leaf_nodes(nodes)        
        storage_context = StorageContext.from_defaults() # 將 index 儲存在記憶體中
        storage_context.docstore.add_documents(nodes)
        # https://docs.llamaindex.ai/en/stable/api_reference/indices/vector_store.html#llama_index.indices.vector_store.base.VectorStoreIndex
        # 這邊我們提供 leaf_nodes 作為 embedding index 的 node，storage context 中則是所有的 nodes 包括整個結構，最後 service_context 是整個 service 用到的相關資源
        automerging_index = VectorStoreIndex(
                leaf_nodes,
                service_context=auto_merging_context,
                storage_context=storage_context
            )
        automerging_index.storage_context.persist(persist_dir=vi_out_path)
    else:
        automerging_index = load_index_from_storage(
            StorageContext.from_defaults(persist_dir=vi_out_path),
            service_context=auto_merging_context,
        )
    
    base_retriever = automerging_index.as_retriever(similarity_top_k=similarity_top_k)
    # https://docs.llamaindex.ai/en/latest/examples/retrievers/auto_merging_retriever.html
    automerging_retriever = AutoMergingRetriever(
        base_retriever, 
        automerging_index.storage_context, 
        verbose=True
    )

    #nodes = automerging_retriever.retrieve("What is the histology of the patient?")
    # for node in nodes:
    #     display_source_node(node, source_length=10000)
    # print("*"*50)
    # base_nodes = base_retriever.retrieve("What is the histology of the patient?")
    # for node in base_nodes:
    #     display_source_node(node, source_length=10000)
    # auto_merging_engines[doc.metadata['date']] = auto_merging_engine
    
    # build summary index
    # summary_index = SummaryIndex(nodes, service_context=service_context)
    
    query_engine_tools = []
    for template in agent_document_idx_templates.keys():
        # define query engines
        # 目前用 base 好像就可以了，用 auto merging 反而做不好??
        # auto_merging_engine = RetrieverQueryEngine.from_args(automerging_retriever, node_postprocessors=[rerank], 
        #                                                      service_context = auto_merging_context)
        auto_merging_engine = RetrieverQueryEngine.from_args(base_retriever, node_postprocessors=[rerank], 
                                                            service_context = auto_merging_context)

        update_query_engine_prompts(agent_document_idx_templates[template], auto_merging_engine)
        prompts_dict = auto_merging_engine.get_prompts()
        display_prompt_dict(prompts_dict)

        # define tools
        query_engine_tools.append(QueryEngineTool(
            query_engine=auto_merging_engine,
            metadata=ToolMetadata(
                name=f"get_{template}",
                description=(f"Useful for answering questions related to {template.replace('_', ' ')} using the medical report dated {doc.metadata['date']}"),
                fn_schema= ToolSchema
            )))

    # summary_query_engine = summary_index.as_query_engine(
    #     response_mode="tree_summarize"
    # )

    # extract a summary
    # if not os.path.exists(summary_out_path):
    #     Path(summary_out_path).parent.mkdir(parents=True, exist_ok=True)
    #     summary = str(
    #         await summary_query_engine.aquery(
    #             "Extract a concise 1-2 line summary of this document"
    #         )
    #     )
    #     pickle.dump(summary, open(summary_out_path, "wb"))
    # else:
    #     summary = pickle.load(open(summary_out_path, "rb"))

    
#         QueryEngineTool(
#             query_engine=summary_query_engine,
#             metadata=ToolMetadata(
#                 name=f"summary_tool_{file_base}",
#                 description=f"Useful for summarization questions",
#             ),
#         ),
#     ]

#     # build agent
#     function_llm = OpenAI(model="gpt-4")
#     agent = OpenAIAgent.from_tools(
#         query_engine_tools,
#         llm=function_llm,
#         verbose=True,
#         system_prompt=f"""\
# You are a specialized agent designed to answer queries about the `{file_base}.html` part of the LlamaIndex docs.
# You must ALWAYS use at least one of the tools provided when answering a question; do NOT rely on prior knowledge.\
# """,
#     )
    agent = ReActAgent.from_tools(query_engine_tools, llm=agent_llm, verbose=True,
                                  system_prompt=f"""You are a specialized agent designed to answer queries about the \
cancer registry coding results based on a patient's medical reports.
You must ALWAYS use at least one of the tools provided when answering a question; do NOT rely on any prior knowledge.\
""")
    return agent, file_id#, summary

from llama_index.node_parser import HierarchicalNodeParser
import os

# https://docs.llamaindex.ai/en/stable/examples/agent/multi_document_agents-v1.html
def build_agents(docs):
    # See https://docs.llamaindex.ai/en/latest/examples/retrievers/auto_merging_retriever.html#
    node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128])

    auto_merging_context = ServiceContext.from_defaults(llm=doc_idx_llm,
                                                        embed_model="local:BAAI/bge-large-en-v1.5",
                                                        node_parser=node_parser)

    # Build agents dictionary
    agents_dict = {}
    extra_info_dict = {}

    # # this is for the baseline
    # all_nodes = []

    for idx, doc in enumerate((docs)):#tqdm(docs)):
        nodes = node_parser.get_nodes_from_documents([doc])
        # all_nodes.extend(nodes)

        # ID will be base + parent
        agent, file_id = build_agent_per_doc(doc, nodes, auto_merging_context, agent_document_idx_templates)#, summary = build_agent_per_doc(doc, nodes, file_id)

        agents_dict[file_id] = agent
        #extra_info_dict[file_id] = {"summary": summary, "nodes": nodes}

    return agents_dict#, extra_info_dict

agents = build_agents(docs)

response = agents['202209000226_2021-08-24'].chat("What is the laterality and primary site of the patient?")
print(str(response))

response = agents['202209000226_2021-08-24'].chat("What is the histology coding of the patient ?")
print(str(response))

In [ ]:
response = agents['202209000226_2021-08-24'].chat("What is the histology coding of the patient ?")

In [ ]:
response = agents['202209000226_2021-08-24'].chat("What is your name?")

In [ ]:
def build_agent_per_doc(doc, nodes, auto_merging_context, agent_document_idx_templates):
    file_path = Path(doc.metadata["file_path"])
    file_id = str(file_path.parent.stem) + "_" + str(file_path.stem)        
    vi_out_path = f"./data/llamaindex_docs/{file_id}"

    class ToolSchema(BaseModel):
        pass
    #summary_out_path = f"./data/llamaindex_docs/{file_id}_summary.pkl"
    if not os.path.exists(vi_out_path):
        leaf_nodes = get_leaf_nodes(nodes)        
        storage_context = StorageContext.from_defaults() # 將 index 儲存在記憶體中
        storage_context.docstore.add_documents(nodes)
        # https://docs.llamaindex.ai/en/stable/api_reference/indices/vector_store.html#llama_index.indices.vector_store.base.VectorStoreIndex
        # 這邊我們提供 leaf_nodes 作為 embedding index 的 node，storage context 中則是所有的 nodes 包括整個結構，最後 service_context 是整個 service 用到的相關資源
        automerging_index = VectorStoreIndex(
                leaf_nodes,
                service_context=auto_merging_context,
                storage_context=storage_context
            )
        automerging_index.storage_context.persist(persist_dir=vi_out_path)
    else:
        automerging_index = load_index_from_storage(
            StorageContext.from_defaults(persist_dir=vi_out_path),
            service_context=auto_merging_context,
        )
    
    base_retriever = automerging_index.as_retriever(similarity_top_k=similarity_top_k)
    # https://docs.llamaindex.ai/en/latest/examples/retrievers/auto_merging_retriever.html
    automerging_retriever = AutoMergingRetriever(
        base_retriever, 
        automerging_index.storage_context, 
        verbose=True
    )

    #nodes = automerging_retriever.retrieve("What is the histology of the patient?")
    # for node in nodes:
    #     display_source_node(node, source_length=10000)
    # print("*"*50)
    # base_nodes = base_retriever.retrieve("What is the histology of the patient?")
    # for node in base_nodes:
    #     display_source_node(node, source_length=10000)
    # auto_merging_engines[doc.metadata['date']] = auto_merging_engine
    
    # build summary index
    # summary_index = SummaryIndex(nodes, service_context=service_context)
    
    query_engine_tools = []
    for template in agent_document_idx_templates.keys():
        # define query engines
        # 目前用 base 好像就可以了，用 auto merging 反而做不好??
        # auto_merging_engine = RetrieverQueryEngine.from_args(automerging_retriever, node_postprocessors=[rerank], 
        #                                                      service_context = auto_merging_context)
        auto_merging_engine = RetrieverQueryEngine.from_args(base_retriever, node_postprocessors=[rerank], 
                                                            service_context = auto_merging_context)

        update_query_engine_prompts(agent_document_idx_templates[template], auto_merging_engine)
        prompts_dict = auto_merging_engine.get_prompts()
        display_prompt_dict(prompts_dict)

        # define tools
        query_engine_tools.append(QueryEngineTool(
            query_engine=auto_merging_engine,
            metadata=ToolMetadata(
                name=f"get_{template}",
                description=(f"Useful for answering questions related to {template.replace('_', ' ')} using the medical report dated {doc.metadata['date']}"),
                fn_schema= ToolSchema
            )))

    # summary_query_engine = summary_index.as_query_engine(
    #     response_mode="tree_summarize"
    # )

    # extract a summary
    # if not os.path.exists(summary_out_path):
    #     Path(summary_out_path).parent.mkdir(parents=True, exist_ok=True)
    #     summary = str(
    #         await summary_query_engine.aquery(
    #             "Extract a concise 1-2 line summary of this document"
    #         )
    #     )
    #     pickle.dump(summary, open(summary_out_path, "wb"))
    # else:
    #     summary = pickle.load(open(summary_out_path, "rb"))

    
#         QueryEngineTool(
#             query_engine=summary_query_engine,
#             metadata=ToolMetadata(
#                 name=f"summary_tool_{file_base}",
#                 description=f"Useful for summarization questions",
#             ),
#         ),
#     ]

#     # build agent
#     function_llm = OpenAI(model="gpt-4")
#     agent = OpenAIAgent.from_tools(
#         query_engine_tools,
#         llm=function_llm,
#         verbose=True,
#         system_prompt=f"""\
# You are a specialized agent designed to answer queries about the `{file_base}.html` part of the LlamaIndex docs.
# You must ALWAYS use at least one of the tools provided when answering a question; do NOT rely on prior knowledge.\
# """,
#     )
    agent = ReActAgent.from_tools(query_engine_tools, llm=agent_llm, verbose=True,
                                  system_prompt=f"""You are a specialized agent designed to answer queries about the \
cancer registry coding results based on a patient's medical reports.
You must ALWAYS use at least one of the tools provided when answering a question; do NOT rely on any prior knowledge.\
""")
    return agent, file_id#, summary

In [ ]:
from llama_index.node_parser import HierarchicalNodeParser
import os

# https://docs.llamaindex.ai/en/stable/examples/agent/multi_document_agents-v1.html
def build_agents(docs):
    # See https://docs.llamaindex.ai/en/latest/examples/retrievers/auto_merging_retriever.html#
    node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128])

    auto_merging_context = ServiceContext.from_defaults(llm=doc_idx_llm,
                                                        embed_model="local:BAAI/bge-large-en-v1.5",
                                                        node_parser=node_parser)

    # Build agents dictionary
    agents_dict = {}
    extra_info_dict = {}

    # # this is for the baseline
    # all_nodes = []

    for idx, doc in enumerate((docs)):#tqdm(docs)):
        nodes = node_parser.get_nodes_from_documents([doc])
        # all_nodes.extend(nodes)

        # ID will be base + parent
        agent, file_id = build_agent_per_doc(doc, nodes, auto_merging_context, agent_document_idx_templates)#, summary = build_agent_per_doc(doc, nodes, file_id)

        agents_dict[file_id] = agent
        #extra_info_dict[file_id] = {"summary": summary, "nodes": nodes}

    return agents_dict#, extra_info_dict

agents = build_agents(docs)

In [ ]:
response = agents['202209000226_2021-08-24'].chat("What is the histology coding of the patient ?")
print(str(response))

In [ ]:
response = agents['202209000226_2021-08-24'].chat("What is the laterality and primary site of the patient?")
print(str(response))

# Build Retriever-Enabled Agent

We build a top-level agent that can orchestrate across the different document agents to answer any user query.

This agent takes in all document agents as tools. This specific agent ReActAgent performs tool retrieval before tool use (unlike a default agent that tries to put all tools in the prompt).

Here we use a top-k retriever, but we should customize the tool retriever method later

In [ ]:
# define tool for each document agent
all_tools = []

for idx, doc in enumerate((docs)):
    wiki_summary = (
        f"This content contains medical records about {wiki_title}. Use"
        f" this tool if you want to answer any questions about {wiki_title}.\n"
    )
    doc_tool = QueryEngineTool(
        query_engine=agents[wiki_title],
        metadata=ToolMetadata(
            name=f"tool_{wiki_title}",
            description=wiki_summary,
        ),
    )
    all_tools.append(doc_tool)

In [ ]:
llama_template_histology_all ='''Your primary task is to accurately curate the histology code for patients diagnosed with lung cancer. This involves meticulously reviewing and analyzing all available medical records of the patient to determine the most appropriate histology code. Your decision should be based on a thorough assessment of clinical findings, pathology reports, and any relevant medical documentation that provides insights into the histological nature of the lung cancer.
When responding, you are required to select the most appropriate option from the following standard codes, which represents different histology types of lung cancer:

- 8550: Use this code if the histology type of the lung cancer case is "acinar cell carcinoma", "acinar adenocarcinoma", "acinar carcinoma", or "acinic cell adenocarcinoma".
- 8551: Use this code if the histology type is "acinar cell cystadenocarcinoma" or "acinar adenocarcinoma of the lung".
- 8041: Use this code if the histology type is "small cell carcinoma, pulmonary type", "small cell neuroendocrine carcinoma", "round cell carcinoma", "reserve cell carcinoma", or "small cell carcinoma, NOS"
- 8070: Use this code if the histology type is "squamous cell carcinoma in situ, NOS", "epidermoid carcinoma in situ, NOS", "intraepidermal carcinoma, NOS", "intraepithelial squamous cell carcinoma", "squamous cell carcinoma, NOS", "squamous cell carcinoma, usual type", "epidermoid carcinoma, NOS", "squamous carcinoma", or "squamous cell epithelioma"

Please ensure your response is limited to one of these specific codes. The code you choose should correspond to the most accurate histological classification of the lung cancer case you are registering. For instance, if the case is diagnosed as 'Acinar adenocarcinoma of the lung', your response must be
{
"explan": "[Your explanation here]",
"histology": "8551"
}'''

llama_template_primary_site_laterality_all ='''Your main responsibility is to accurately determine the primary site and laterality codes for patients diagnosed with lung cancer. This task requires a detailed and careful review of all available medical records for each patient. You are expected to analyze clinical findings, pathology reports, and any other relevant medical documentation that offers insights into the specific location and laterality of the lung cancer.
Choose from the following standard codes for lung cancer sites and laterality:

Primary site codes:
- C339: Trachea
- C340: Main bronchus
- C341: Upper lobe, lung
- C342: Middle lobe, lung
- C343: Lower lobe, lung
- C348: Overlapping lesion of lung
- C349: Lung NOS (Not Otherwise Specified)

Laterality codes:
- 0: Cancer does not originate from a paired organ.
- 1: Primary origin of the cancer is on the right side.
- 2: Primary origin of the cancer is on the left side.

Your response must be a valid JSON object containing the following keys:
-'primary site': A string containing the code for the primary site.
-'laterality': A string containing the code for laterality.

Ensure your response is limited to the provided options for primary site and laterality. For instance, if the primary site and laterality are in the 'upper lobe' of the 'right' lung, your JSON response should be:
{
"explan": "[Your explanation here]",
"primary site": "C341",
"laterality": "1"
}'''

#llama_message = {'role': 'user', 'content': example_path_text+"\nCurate the code for 'Laterality' of the tumor if available"}
#llama_message = {'role': 'user', 'content': example_path_text+"\nCurate the code for 'Histology' of the lung cancer if available"}

                  ]
llama_templates_all = [#{'role': 'user', 'content': llama_template_laterality}, {'role': 'user', 'content': llama_template_primary_site},
                  #  {'role': 'user', 'content': llama_template_path_TNM},
                #    {'role': 'user', 'content': llama_template_behavior}, {'role': 'user', 'content': llama_template_node_examine},
                #    {'role': 'user', 'content': llama_template_node_positive},
                    {'role': 'user', 'content': llama_template_histology_all},
                    {'role': 'user', 'content': llama_template_primary_site_laterality_all}
                  ]
llama_templates_all

In [ ]:
eval_questions = []
for i in range(len(llama_templates_all)):
    eval_questions.append(llama_templates_all[i]['content'])#format_dialogue_prompt([llama_templates[i]], system_message))
print(eval_questions)
answer_relevance_golden_set = [#{"query": eval_questions[0], "response": "2"}, 
                               #{"query": eval_questions[1], "response": "C343"},
                                #{"query": eval_questions[0], "response": "PATH_T: 2\nPATH_N: 0"},
                              #   {"query": eval_questions[1], "response": "3"},
                              #   {"query": eval_questions[2], "response": "2"},
                              #    {"query": eval_questions[3], "response": "0"},                                 
                                {"query": eval_questions[0], "response": "8551"},
                                {"query": eval_questions[1], "response": "PS: C343\nL: 2"},
                                ]
answer_relevance_golden_set

In [ ]:
from llama_index import VectorStoreIndex
from llama_index.agent import ReActAgent
from llama_index.tools import QueryEngineTool, ToolMetadata

index = VectorStoreIndex.from_documents(docs, llm = agent_llm, service_context=agent_service_context)
engine = index.as_query_engine(similarity_top_k=6)

query_engine_tools = [
    QueryEngineTool(
        query_engine=engine,
        metadata=ToolMetadata(
            name="get_report",
            description=(
                "Report provides information about the patient"
            ),
        ),
    ),
]

agent = ReActAgent.from_tools(query_engine_tools, llm=agent_llm, verbose=True)

response = agent.chat("What was the histology of the patient?")
print(str(response))

# Document Agent

## Chat Templating

See https://huggingface.co/docs/transformers/chat_templating

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

chat = [
  {"role": "user", "content": "Hello, how are you?"},
  {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
  {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

print(tokenizer.default_chat_template)
print(tokenizer.apply_chat_template(chat, tokenize=False))

chat = [
  {"role": "user", "content": "Hello, how are you?"},
   {"role": "user", "content": "I'd like to show off how chat templating works!"},
    {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
]

# cause error
print(tokenizer.apply_chat_template(chat, tokenize=False))

## Define Document Agent
See https://colab.research.google.com/drive/12k7RVPmGpPfFNXY6fGvqtMoAVOtjz44o?usp=sharing
by using messages_to_prompt to fix the error of "Conversation roles must alternate user/assistant/user/assistant/..."

In [ ]:
from llama_index.llms.llama_utils import messages_to_prompt
??messages_to_prompt

In [ ]:
for idx, node in enumerate(file_2_codings['2021-10-15']['primary_site_laterality'].source_nodes):
    print(f'{idx}: {node.text}')

In [ ]:
for idx, node in enumerate(file_2_codings['2021-10-15']['histology'].source_nodes):
    print(f'{idx}: {node.text}')

In [ ]:
file_2_codings

In [ ]:
file_2_index

In [ ]:
file_2_engine

In [ ]:
from llama_index.output_parsers.utils import parse_json_markdown
from llama_index.question_gen.types import SubQuestion
from llama_index.types import BaseOutputParser
from typing import Any
from llama_index.output_parsers.base import StructuredOutput

class SubQuestionOutputParser(BaseOutputParser):
    def parse(self, output: str) -> Any:
        json_dict = parse_json_markdown(output)
        if not json_dict:
            raise ValueError(f"No valid JSON found in output: {output}")

        sub_questions = [SubQuestion.parse_obj(item) for item in json_dict]
        return StructuredOutput(raw_output=output, parsed_output=sub_questions)

    def format(self, prompt_template: str) -> str:
        return prompt_template

In [ ]:
from typing import List, Optional, Sequence, cast

from llama_index.llm_predictor.base import BaseLLMPredictor
from llama_index.output_parsers.base import StructuredOutput
from llama_index.prompts.base import BasePromptTemplate, PromptTemplate
from llama_index.prompts.mixin import PromptDictType
from llama_index.prompts.prompt_type import PromptType
#from llama_index.question_gen.output_parser import SubQuestionOutputParser
from llama_index.question_gen.prompts import (
    DEFAULT_SUB_QUESTION_PROMPT_TMPL,
    build_tools_text,
)
from llama_index.question_gen.types import BaseQuestionGenerator, SubQuestion
from llama_index.schema import QueryBundle
from llama_index.service_context import ServiceContext
from llama_index.tools.types import ToolMetadata
from llama_index.types import BaseOutputParser


class LLMCancerRegistryQuestionGenerator(BaseQuestionGenerator):
    def __init__(
        self,
        llm_predictor: BaseLLMPredictor,
        prompt: BasePromptTemplate,
    ) -> None:
        self._llm_predictor = llm_predictor
        self._prompt = prompt

        if self._prompt.output_parser is None:
            raise ValueError("Prompt should have output parser.")

    @classmethod
    def from_defaults(
        cls,
        service_context: Optional[ServiceContext] = None,
        prompt_template_str: Optional[str] = None,
        output_parser: Optional[BaseOutputParser] = None,
    ) -> "LLMCancerRegistryQuestionGenerator":
        # optionally initialize defaults
        service_context = service_context or ServiceContext.from_defaults()
        prompt_template_str = prompt_template_str or DEFAULT_SUB_QUESTION_PROMPT_TMPL
        output_parser = output_parser or SubQuestionOutputParser()

        # construct prompt
        prompt = PromptTemplate(
            template=prompt_template_str,
            output_parser=output_parser,
            prompt_type=PromptType.SUB_QUESTION,
        )
        return cls(service_context.llm_predictor, prompt)

    def _get_prompts(self) -> PromptDictType:
        """Get prompts."""
        return {"question_gen_prompt": self._prompt}

    def _update_prompts(self, prompts: PromptDictType) -> None:
        """Update prompts."""
        if "question_gen_prompt" in prompts:
            self._prompt = prompts["question_gen_prompt"]

    def generate(
        self, tools: Sequence[ToolMetadata], query: QueryBundle
    ) -> List[SubQuestion]:
        tools_str = build_tools_text(tools)
        query_str = query.query_str
        prediction = self._llm_predictor.predict(
            prompt=self._prompt,
            tools_str=tools_str,
            query_str=query_str,
        )

        assert self._prompt.output_parser is not None
        parse = self._prompt.output_parser.parse(prediction)
        parse = cast(StructuredOutput, parse)
        return parse.parsed_output

    async def agenerate(
        self, tools: Sequence[ToolMetadata], query: QueryBundle
    ) -> List[SubQuestion]:
        tools_str = build_tools_text(tools)
        query_str = query.query_str
        prediction = await self._llm_predictor.apredict(
            prompt=self._prompt,
            tools_str=tools_str,
            query_str=query_str,
        )

        assert self._prompt.output_parser is not None
        parse = self._prompt.output_parser.parse(prediction)
        parse = cast(StructuredOutput, parse)
        return parse.parsed_output


In [ ]:
file_2_engine.items()

In [ ]:
import asyncio
import logging
from typing import List, Optional, Sequence, cast

from llama_index.async_utils import run_async_tasks
from llama_index.bridge.pydantic import BaseModel, Field
from llama_index.callbacks.base import CallbackManager
from llama_index.callbacks.schema import CBEventType, EventPayload
from llama_index.core import BaseQueryEngine
from llama_index.prompts.mixin import PromptMixinType
from llama_index.question_gen.llm_generators import LLMQuestionGenerator
from llama_index.question_gen.openai_generator import OpenAIQuestionGenerator
from llama_index.question_gen.types import BaseQuestionGenerator, SubQuestion
from llama_index.response.schema import RESPONSE_TYPE
from llama_index.response_synthesizers import BaseSynthesizer, get_response_synthesizer
from llama_index.schema import NodeWithScore, QueryBundle, TextNode
from llama_index.service_context import ServiceContext
from llama_index.tools.query_engine import QueryEngineTool
from llama_index.utils import get_color_mapping, print_text

logger = logging.getLogger(__name__)


class SubQuestionAnswerPair(BaseModel):
    """
    Pair of the sub question and optionally its answer (if its been answered yet).
    """

    sub_q: SubQuestion
    answer: Optional[str] = None
    sources: List[NodeWithScore] = Field(default_factory=list)


class MySubQuestionQueryEngine(BaseQueryEngine):
    """Sub question query engine.

    A query engine that breaks down a complex query (e.g. compare and contrast) into
        many sub questions and their target query engine for execution.
        After executing all sub questions, all responses are gathered and sent to
        response synthesizer to produce the final response.

    Args:
        question_gen (BaseQuestionGenerator): A module for generating sub questions
            given a complex question and tools.
        response_synthesizer (BaseSynthesizer): A response synthesizer for
            generating the final response
        query_engine_tools (Sequence[QueryEngineTool]): Tools to answer the
            sub questions.
        verbose (bool): whether to print intermediate questions and answers.
            Defaults to True
        use_async (bool): whether to execute the sub questions with asyncio.
            Defaults to True
    """

    def __init__(
        self,
        question_gen: BaseQuestionGenerator,
        response_synthesizer: BaseSynthesizer,
        query_engine_tools: Sequence[QueryEngineTool],
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = True,
        use_async: bool = False,
    ) -> None:
        self._question_gen = question_gen
        self._response_synthesizer = response_synthesizer
        self._metadatas = [x.metadata for x in query_engine_tools]
        self._query_engines = {
            tool.metadata.name: tool.query_engine for tool in query_engine_tools
        }
        self._verbose = verbose
        self._use_async = use_async
        super().__init__(callback_manager)

    def _get_prompt_modules(self) -> PromptMixinType:
        """Get prompt sub-modules."""
        return {
            "question_gen": self._question_gen,
            "response_synthesizer": self._response_synthesizer,
        }

    @classmethod
    def from_defaults(
        cls,
        query_engine_tools: Sequence[QueryEngineTool],
        question_gen: Optional[BaseQuestionGenerator] = None,
        response_synthesizer: Optional[BaseSynthesizer] = None,
        service_context: Optional[ServiceContext] = None,
        verbose: bool = True,
        use_async: bool = True,
    ) -> "SubQuestionQueryEngine":
        callback_manager = None
        if service_context is not None:
            callback_manager = service_context.callback_manager
        elif len(query_engine_tools) > 0:
            callback_manager = query_engine_tools[0].query_engine.callback_manager

        if question_gen is None:
            if service_context is None:
                # use default openai model that supports function calling API
                question_gen = OpenAIQuestionGenerator.from_defaults()
            else:
                # try to use OpenAI function calling based question generator.
                # if incompatible, use general LLM question generator
                try:
                    question_gen = OpenAIQuestionGenerator.from_defaults(
                        llm=service_context.llm
                    )
                except ValueError:
                    question_gen = LLMCancerRegistryQuestionGenerator.from_defaults(
                        service_context=service_context
                    )

        synth = response_synthesizer or get_response_synthesizer(
            callback_manager=callback_manager,
            service_context=service_context,
            use_async=use_async,
        )

        return cls(
            question_gen,
            synth,
            query_engine_tools,
            callback_manager=callback_manager,
            verbose=verbose,
            use_async=use_async,
        )

    def _query(self, query_bundle: QueryBundle) -> RESPONSE_TYPE:
        with self.callback_manager.event(
            CBEventType.QUERY, payload={EventPayload.QUERY_STR: query_bundle.query_str}
        ) as query_event:
            sub_questions = self._question_gen.generate(self._metadatas, query_bundle)

            colors = get_color_mapping([str(i) for i in range(len(sub_questions))])

            if self._verbose:
                print_text(f"Generated {len(sub_questions)} sub questions.\n")

            if self._use_async:
                tasks = [
                    self._aquery_subq(sub_q, color=colors[str(ind)])
                    for ind, sub_q in enumerate(sub_questions)
                ]

                qa_pairs_all = run_async_tasks(tasks)
                qa_pairs_all = cast(List[Optional[SubQuestionAnswerPair]], qa_pairs_all)
            else:
                qa_pairs_all = [
                    self._query_subq(sub_q, color=colors[str(ind)])
                    for ind, sub_q in enumerate(sub_questions)
                ]

            # filter out sub questions that failed
            qa_pairs: List[SubQuestionAnswerPair] = list(filter(None, qa_pairs_all))

            nodes = [self._construct_node(pair) for pair in qa_pairs]

            source_nodes = [node for qa_pair in qa_pairs for node in qa_pair.sources]
            response = self._response_synthesizer.synthesize(
                query=query_bundle,
                nodes=nodes,
                additional_source_nodes=source_nodes,
            )

            query_event.on_end(payload={EventPayload.RESPONSE: response})

        return response

    async def _aquery(self, query_bundle: QueryBundle) -> RESPONSE_TYPE:
        with self.callback_manager.event(
            CBEventType.QUERY, payload={EventPayload.QUERY_STR: query_bundle.query_str}
        ) as query_event:
            sub_questions = await self._question_gen.agenerate(
                self._metadatas, query_bundle
            )

            colors = get_color_mapping([str(i) for i in range(len(sub_questions))])

            if self._verbose:
                print_text(f"Generated {len(sub_questions)} sub questions.\n")

            tasks = [
                self._aquery_subq(sub_q, color=colors[str(ind)])
                for ind, sub_q in enumerate(sub_questions)
            ]

            qa_pairs_all = await asyncio.gather(*tasks)
            qa_pairs_all = cast(List[Optional[SubQuestionAnswerPair]], qa_pairs_all)

            # filter out sub questions that failed
            qa_pairs: List[SubQuestionAnswerPair] = list(filter(None, qa_pairs_all))

            nodes = [self._construct_node(pair) for pair in qa_pairs]

            source_nodes = [node for qa_pair in qa_pairs for node in qa_pair.sources]
            response = await self._response_synthesizer.asynthesize(
                query=query_bundle,
                nodes=nodes,
                additional_source_nodes=source_nodes,
            )

            query_event.on_end(payload={EventPayload.RESPONSE: response})

        return response

    def _construct_node(self, qa_pair: SubQuestionAnswerPair) -> NodeWithScore:
        node_text = (
            f"Sub question: {qa_pair.sub_q.sub_question}\nResponse: {qa_pair.answer}"
        )
        return NodeWithScore(node=TextNode(text=node_text))

    async def _aquery_subq(
        self, sub_q: SubQuestion, color: Optional[str] = None
    ) -> Optional[SubQuestionAnswerPair]:
        try:
            with self.callback_manager.event(
                CBEventType.SUB_QUESTION,
                payload={EventPayload.SUB_QUESTION: SubQuestionAnswerPair(sub_q=sub_q)},
            ) as event:
                question = sub_q.sub_question
                query_engine = self._query_engines[sub_q.tool_name]

                if self._verbose:
                    print_text(f"[{sub_q.tool_name}] Q: {question}\n", color=color)

                response = await query_engine.aquery(question)
                response_text = str(response)

                if self._verbose:
                    print_text(f"[{sub_q.tool_name}] A: {response_text}\n", color=color)

                qa_pair = SubQuestionAnswerPair(
                    sub_q=sub_q, answer=response_text, sources=response.source_nodes
                )

                event.on_end(payload={EventPayload.SUB_QUESTION: qa_pair})

            return qa_pair
        except ValueError:
            logger.warning(f"[{sub_q.tool_name}] Failed to run {question}")
            return None
        except KeyError:
            logger.warning(f"[{sub_q.tool_name}] Failed to run {question}")
            return None

    def _query_subq(
        self, sub_q: SubQuestion, color: Optional[str] = None
    ) -> Optional[SubQuestionAnswerPair]:
        try:
            with self.callback_manager.event(
                CBEventType.SUB_QUESTION,
                payload={EventPayload.SUB_QUESTION: SubQuestionAnswerPair(sub_q=sub_q)},
            ) as event:
                question = sub_q.sub_question
                query_engine = self._query_engines[sub_q.tool_name]

                if self._verbose:
                    print_text(f"[{sub_q.tool_name}] Q: {question}\n", color=color)

                response = query_engine.query(question)
                response_text = str(response)

                if self._verbose:
                    print_text(f"[{sub_q.tool_name}] A: {response_text}\n", color=color)

                qa_pair = SubQuestionAnswerPair(
                    sub_q=sub_q, answer=response_text, sources=response.source_nodes
                )

                event.on_end(payload={EventPayload.SUB_QUESTION: qa_pair})

            return qa_pair
        except ValueError:
            logger.warning(f"[{sub_q.tool_name}] Failed to run {question}")
            return None
        except KeyError:
            logger.warning(f"[{sub_q.tool_name}] Failed to run {question}")
            return None


from llama_index.tools import QueryEngineTool, ToolMetadata
from llama_index.query_engine import SubQuestionQueryEngine
import json

# Define a List of QueryEngineTools wrapping all individual pdf file indices 
query_engine_tools = []
for title, engine in file_2_engine.items():
    query_engine_tools.append(QueryEngineTool(query_engine=engine,
    metadata=ToolMetadata(
        #name=re.sub(r"_\d", "", title).replace("-", "_"),
        name="histology",
        description='Patient\'s report at {0}. Key content: {1}'.format(re.sub(r"_\d", "", title),
                                                           json.loads(file_2_codings['2021-10-15']['histology'].response)["explan"])
        )))    
    query_engine_tools.append(QueryEngineTool(query_engine=engine,
        metadata=ToolMetadata(
            #name=re.sub(r"_\d", "", title).replace("-", "_"),
            name="primary site",
            description='Patient\'s report at {0}. Key content: {1}'.format(re.sub(r"_\d", "", title),
                                                            json.loads(file_2_codings['2021-10-15']['primary_site_laterality'].response)["explan"]))))



# Initialize a multi-index query engine based on all QueryEngineTools 
multi_index_query_engine = MySubQuestionQueryEngine.from_defaults(query_engine_tools=query_engine_tools, service_context=auto_merging_context, use_async=False)

In [ ]:
file_2_codings['2021-10-15']

In [ ]:
def run_queries(queries, query_engine,  tru_recorder = None):
    responses = []
    for q in queries:
        print('='*50+f'\nProcessing query "{q}"...')
        response = None
        if tru_recorder is not None:
            with tru_recorder as recording:
                response = query_engine.query(q)
        else:
            response = query_engine.query(q)
        print(response)
        responses.append(response)
    return responses


run_queries(eval_questions, multi_index_query_engine)     

## Put it all together

In [ ]:
import os

from llama_index import (
    ServiceContext,
    StorageContext,
    VectorStoreIndex,
    load_index_from_storage,
)
from llama_index.node_parser import HierarchicalNodeParser
from llama_index.node_parser import get_leaf_nodes
from llama_index import StorageContext, load_index_from_storage
from llama_index.retrievers import AutoMergingRetriever
from llama_index.indices.postprocessor import SentenceTransformerRerank
from llama_index.query_engine import RetrieverQueryEngine


def build_automerging_index(
    documents,
    llm,
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="merging_index",
    chunk_sizes=None,
):
    chunk_sizes = chunk_sizes or [2048, 512, 128]
    node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=chunk_sizes)
    nodes = node_parser.get_nodes_from_documents(documents)
    leaf_nodes = get_leaf_nodes(nodes)
    merging_context = ServiceContext.from_defaults(
        llm=llm,
        embed_model=embed_model,
    )
    storage_context = StorageContext.from_defaults()
    storage_context.docstore.add_documents(nodes)

    if not os.path.exists(save_dir):
        automerging_index = VectorStoreIndex(
            leaf_nodes, storage_context=storage_context, service_context=merging_context
        )
        automerging_index.storage_context.persist(persist_dir=save_dir)
    else:
        automerging_index = load_index_from_storage(
            StorageContext.from_defaults(persist_dir=save_dir),
            service_context=merging_context,
        )
    return automerging_index, merging_context


def get_automerging_query_engine(
    automerging_index,
    service_context,
    similarity_top_k=12,
    rerank_top_n=6,    
):
    base_retriever = automerging_index.as_retriever(similarity_top_k=similarity_top_k)
    retriever = AutoMergingRetriever(
        base_retriever, automerging_index.storage_context, verbose=True
    )
    rerank = SentenceTransformerRerank(
        top_n=rerank_top_n, model="BAAI/bge-reranker-base"
    )
    auto_merging_engine = RetrieverQueryEngine.from_args(
        retriever, node_postprocessors=[rerank], service_context= context
    )
    return auto_merging_engine

index, context = build_automerging_index(
    [document],
    llm=llm,
    save_dir="./merging_index",
)
query_engine = get_automerging_query_engine(index, similarity_top_k=6, service_context=context)

In [ ]:
from llama_index.response.notebook_utils import display_response

auto_merging_response = query_engine.query(
    eval_questions[1]
)

display_response(auto_merging_response)

## Truelens Evaluation

In [ ]:
from trulens_eval import Tru

Tru().reset_database()

### Two layer first

In [ ]:
auto_merging_index_0, context = build_automerging_index(
    documents,
    llm=llm,
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="merging_index_2layer",
    chunk_sizes=[2048,512],
)

In [ ]:
auto_merging_engine_0 = get_automerging_query_engine(
    auto_merging_index_0,
    context,
    similarity_top_k=12,
    rerank_top_n=6,
)

In [ ]:
tru_recorder_0 = get_prebuilt_trulens_recorder(auto_merging_engine_0, app_id=f'auto-merging engine-2layer', \
                                             answer_relevance_golden_set=answer_relevance_golden_set,\
                                             context_relevance_golden_set=answer_relevance_golden_set)

In [ ]:
run_queries(eval_questions, auto_merging_engine_0, tru_recorder_0)

In [ ]:
from trulens_eval import Tru

Tru().get_leaderboard(app_ids=[])

In [ ]:
Tru().run_dashboard()

## Three layers

In [ ]:
auto_merging_index_1, context = build_automerging_index(
    documents,
    llm=llm,
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="merging_index_3layer",
    chunk_sizes=[2048,512,128],
)

In [ ]:
auto_merging_engine_1 = get_automerging_query_engine(
    auto_merging_index_1,
    context,
    similarity_top_k=12,
    rerank_top_n=6,
)

tru_recorder = get_prebuilt_trulens_recorder(query_engine, app_id=f'auto-merging engine-3layer', \
                                             answer_relevance_golden_set=answer_relevance_golden_set,\
                                             context_relevance_golden_set=answer_relevance_golden_set)

In [ ]:
run_queries(eval_questions, auto_merging_engine_1, tru_recorder)

In [ ]:
from trulens_eval import Tru

Tru().get_leaderboard(app_ids=[])
Tru().run_dashboard()